In [41]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from tqdm import tqdm
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [ ]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

In [6]:
import os
import json
import pandas as pd
import numpy as np
import logging
import re
from typing import List, Dict, Any
import fasttext
import anthropic
import nest_asyncio
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio

pd.options.mode.chained_assignment = None


In [ ]:
null_status = {}
for idx,col in enumerate(df.columns):    
    null_status[col] = round(df[col].notna().sum() / len(df),2)

pd.DataFrame(null_status,index=[0]).T

In [ ]:
class MedicalTextClassifier:
    def __init__(self, api_key:str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.sample = _random_sample()
    
    async def make_api_call(self, prompt: str, semaphore: asyncio.Semahore) -> list(Dict):
        async with semaphore:
            response = await asyncio.to_thread(
                self.client.messages.create,
                model=Config.MODEL_NAME,
                max_tokens=Config.MAX_TOKENS,
                temperature=Config.TEMPERATURE,
                top_p=Config.TOP_P,
                messages=[{"role":"user", "content":prompt}]       
            )
            
            content = response.content[0].text
            result = self._validate_and_parse_json(content)
            if not result:
                raise ValueError("Invalid JSON response")
            return result
    
    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        try:
            array_pattern = r'\[(?:[^[\]]*|\[(?:[^[\]]*|\[[^[\]]*\])*\])*\]'
            matches = list(re.finditer(array_pattern, content))
            if not matches:
                return []
            
            longest_match = max(matches, key=lambda match : len(match.group()))
            potential_json = longest_match.group()
            parsed = json.loads(potential_json)
            if isinstance(parsed, list):
                for item in parsed:
                    item.pop('finish_reason',None)
                    for key, value in item.items():
                        if isinstance(value, dict):
                            value.pop('finish_reson',None)
                return parsed
            return []
        
        except json.JSONDecodeError:
            logger.error(f"JSON parsing failed. Response content: {content[:500]}")
            return []
            
    def _random_sample(self, df: pd.DataFrame, api_key:str, content: str) -> List[Dict]:
        df.
        return 
    
    
    async def _generate_prompt(self, texts: List[str], semaphore : asyncio.Semaphore) -> List[Dict]:
        prompt = f""" 
        
        """
        return await self._make_api_call(prompt, semaphore)
        

In [ ]:
import asyncio
import json
import re
import logging
import pandas as pd
import anthropic  # anthropic 패키지가 설치되어 있어야 합니다.

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# API 호출 및 모델 관련 설정
class Config:
    SEMAPHORE_LIMIT = 5
    MODEL_NAME = "claude-3-5-haiku-20241022"  # 실제 사용하는 모델명으로 변경하세요.
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    TOP_P = 1

class MedicalCategoryRecommender:
    def __init__(self, api_key: str, df: pd.DataFrame):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.df = df

    async def make_api_call(self, prompt: str) -> dict:
        async with self.semaphore:
            response = await asyncio.to_thread(
                self.client.messages.create,
                model=Config.MODEL_NAME,
                max_tokens=Config.MAX_TOKENS,
                temperature=Config.TEMPERATURE,
                top_p=Config.TOP_P,
                messages=[{"role": "user", "content": prompt}]
            )
            content = response.content[0].text
            result = self._validate_and_parse_json(content)
            if not result:
                raise ValueError("Invalid JSON response")
            return result

    def _validate_and_parse_json(self, content: str) -> dict:
        """
        응답 텍스트에서 JSON 객체를 추출합니다.
        """
        try:
            # 우선 직접 파싱 시도
            json_obj = json.loads(content)
            return json_obj
        except json.JSONDecodeError:
            # JSON 파싱에 실패하면 정규표현식으로 추출 시도
            pattern = r'\{.*\}'
            match = re.search(pattern, content, re.DOTALL)
            if match:
                try:
                    json_obj = json.loads(match.group())
                    return json_obj
                except json.JSONDecodeError:
                    logger.error("정규표현식으로 추출한 JSON 파싱 실패")
                    return {}
            logger.error("JSON 파싱 실패. Content: " + content[:500])
            return {}

    def _random_sample(self, column_data: pd.Series, sample_size: int = 10) -> list:
        """
        각 컬럼에서 Null이 아닌 데이터만 대상으로 랜덤하게 sample_size 개의 데이터를 추출합니다.
        """
        # Null이 아닌 데이터만 필터링
        valid_data = column_data.dropna()
        
        # 유효한 데이터가 없는 경우
        if len(valid_data) == 0:
            logger.warning(f"컬럼 '{column_data.name}'에 유효한 데이터가 없습니다.")
            return []
        
        # 유효한 데이터가 sample_size보다 적은 경우
        if len(valid_data) < sample_size:
            logger.info(f"컬럼 '{column_data.name}'의 유효한 데이터가 {len(valid_data)}개로, 요청된 sample_size({sample_size})보다 적습니다.")
            return valid_data.tolist()
        
        # 요청된 sample_size만큼 랜덤 샘플링
        return valid_data.sample(n=sample_size, random_state=42).tolist()

    def _generate_prompt_for_column(self, column_name: str, sample_data: list) -> str:
        """
        컬럼명과 해당 컬럼의 샘플 데이터를 기반으로 카테고리 추천 프롬프트를 생성합니다.
        """
        joined_data = "\n".join([str(item) for item in sample_data])
        prompt = f"""
            당신은 데이터 분석 전문가이자, 카테고리 분류 전문가입니다.
            아래는 '{column_name}' 컬럼의 샘플 데이터입니다:
            {joined_data}

            이 데이터를 바탕으로 '{column_name}' 컬럼을 효과적으로 분류할 수 있는 유의미한 카테고리(또는 범주)를 추천해 주세요.
            각 카테고리에 대해 간단한 설명도 함께 제공해 주시고, 반드시 아래 JSON 형식으로 응답해 주세요.

            응답 예시:
            {{
            "column": "{column_name}",
            "categories": [
                {{
                    "name": "예시 카테고리 1",
                    "description": "카테고리 1에 대한 설명"
                }},
                {{
                    "name": "예시 카테고리 2",
                    "description": "카테고리 2에 대한 설명"
                }}
                // 필요에 따라 더 추가
            ]
            }}
        """
        return prompt

    async def _get_recommendation(self, column: str, prompt: str):
        """
        특정 컬럼에 대해 API 호출하여 추천 결과를 받습니다.
        """
        try:
            result = await self.make_api_call(prompt)
            return (column, result)
        except Exception as e:
            logger.error(f"컬럼 {column} 처리 중 오류 발생: {e}")
            return (column, {})


    async def recommend_categories_for_all_columns(self) -> dict:
        from tqdm import tqdm  # 이렇게 수정
        """
        DataFrame의 모든 컬럼에 대해 추천 카테고리를 받아옵니다.
        """
        recommendations = {}
        tasks = []
        
        for column in tqdm(self.df.columns, desc="컬럼 처리 중"):  # tqdm.tqdm 대신 tqdm 사용
            # 샘플 데이터 추출
            sample_data = self._random_sample(self.df[column], sample_size=10)
            
            # 유효한 샘플이 없는 경우 스킵
            if not sample_data:
                logger.warning(f"컬럼 '{column}'에서 유효한 샘플을 추출할 수 없습니다.")
                continue
                
            try:
                prompt = self._generate_prompt_for_column(column, sample_data)
                tasks.append(self._get_recommendation(column, prompt))
                
            except Exception as e:
                logger.error(f"'{column}' 컬럼 처리 준비 중 오류 발생: {str(e)}")
                continue
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        for col, result in results:
            if isinstance(result, Exception):
                logger.error(f"'{col}' 컬럼 처리 중 오류 발생: {str(result)}")
                continue
            recommendations[col] = result
        
        return recommendations

# 예제 실행 코드
async def main():
    # 예시 DataFrame 생성: 실제 데이터로 대체하세요.
    columns = ['날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
               'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
               'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
               'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
               'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
               'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']
    
    # 각 컬럼에 대해 간단한 예시 데이터 생성
    # 실제 API 키를 입력하세요.
    recommender = MedicalCategoryRecommender(api_key, df)
    recommendations = await recommender.recommend_categories_for_all_columns()
    
    # 추천 결과 출력 (각 컬럼에 대해 추천받은 카테고리 JSON)
    print(json.dumps(recommendations, indent=2, ensure_ascii=False))
    
async def main():
    # ... 기존 코드 ...
    recommender = MedicalCategoryRecommender(api_key, df)
    recommendations = await recommender.recommend_categories_for_all_columns()
    
    # 결과를 데이터프레임으로 변환
    categories_data = []
    for column, result in recommendations.items():
        if 'categories' in result:
            for category in result['categories']:
                categories_data.append({
                    'column': column,
                    'category_name': category['name'],
                    'description': category['description']
                })
    
    # 데이터프레임 생성
    categories_df = pd.DataFrame(categories_data)
    
    # 결과 출력
    print("카테고리 추천 결과:")
    display(categories_df)
    
    # JSON 형식으로도 저장
    categories_df.to_json('category_recommendations.json', 
                         orient='records', 
                         force_ascii=False, 
                         indent=2)
    return categories_df

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    categories_df = await main()


In [46]:
# pd.set_option('display.max_rows', None)
categories_df.to_excel('../data/category.xlsx')